<a href="https://www.kaggle.com/code/somnathg25ait2107/mlops-project-gr34?scriptVersionId=326684904" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install -q transformers datasets wandb huggingface_hub scikit-learn

In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import wandb
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

In [3]:
secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
login(token=secrets.get_secret("HF_TOKEN"))
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
MODEL_NAME = "distilbert-base-uncased"
HF_REPO = "somnathchakraborty/distilbert-imdb-sentiment"
VERSION = "v1"

EPOCHS = 4
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.02
WARMUP_STEPS = 50

In [5]:
dataset = load_dataset("stanfordnlp/imdb")

print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"Features: {dataset['train'].features}")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train samples: 25000
Test samples: 25000
Features: {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


In [6]:
def clean_text(text):
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


train_df = dataset['train'].to_pandas()
train_df['text'] = train_df['text'].apply(clean_text)
train_df = train_df.sample(n=5000, random_state=42).reset_index(drop=True)
train_df = train_df.drop_duplicates(subset='text').reset_index(drop=True)

test_df = dataset['test'].to_pandas()
test_df['text'] = test_df['text'].apply(clean_text)
test_df = test_df.sample(n=1000, random_state=42).reset_index(drop=True)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"\nClass distribution:\n{train_df['label'].value_counts()}")
print(f"\nMissing values: {train_df.isnull().sum().to_dict()}")

Train: 4995, Test: 1000

Class distribution:
label
0    2511
1    2484
Name: count, dtype: int64

Missing values: {'text': 0, 'label': 0}


In [7]:
id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_data(df, max_length=512):
    encodings = tokenizer(
        df['text'].tolist(),
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors='pt',
    )
    return Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': torch.tensor(df['label'].values),
    })


train_dataset = tokenize_data(train_df)
test_dataset = tokenize_data(test_df)

print(f"Train dataset: {len(train_dataset)}")
print(f"Test dataset: {len(test_dataset)}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train dataset: 4995
Test dataset: 1000


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

print(f"Model parameters: {model.num_parameters():,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 66,955,010


In [10]:
wandb.init(
    entity="ashish-iit-jodhpur-25ait2051",
    project="mlops-assignment3",
    name=f"run-{VERSION}",
    config={
        "model": MODEL_NAME,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_steps": WARMUP_STEPS,
        "version": VERSION,
        "platform": "Kaggle",
    },
)

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260612_181617-h60fmclz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run run-v1
wandb: ⭐️ View project at https://wandb.ai/ashish-iit-jodhpur-25ait2051/mlops-assignment3
wandb: 🚀 View run at https://wandb.ai/ashish-iit-jodhpur-25ait2051/mlops-assignment3/runs/h60fmclz


In [11]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name=f"run-{VERSION}",
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.593828,0.877000,0.876008
2,No log,0.531643,0.913000,0.913010
3,No log,0.680592,0.904000,0.903984
4,0.406285,0.826222,0.911000,0.910993


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=628, training_loss=0.3382216228800974, metrics={'train_runtime': 590.8387, 'train_samples_per_second': 33.816, 'train_steps_per_second': 1.063, 'total_flos': 2646698625146880.0, 'train_loss': 0.3382216228800974, 'epoch': 4.0})

In [12]:
results = trainer.evaluate()
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1: {results['eval_f1']:.4f}")
print(f"Loss: {results['eval_loss']:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Accuracy: 0.9130
F1: 0.9130
Loss: 0.5316


In [13]:
current_accuracy = results['eval_accuracy']

try:
    from huggingface_hub import model_info
    info = model_info(HF_REPO)
    existing_accuracy = float(info.card_data.get("eval_accuracy", 0)) if info.card_data else 0
except Exception:
    existing_accuracy = 0

print(f"Current run accuracy: {current_accuracy:.4f}")
print(f"Existing HF model accuracy: {existing_accuracy:.4f}")

if current_accuracy > existing_accuracy:
    model.push_to_hub(HF_REPO, commit_message=f"run-{VERSION} accuracy={current_accuracy:.4f}")
    tokenizer.push_to_hub(HF_REPO)

    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_file(
        path_or_fileobj=json.dumps({"eval_accuracy": current_accuracy}).encode(),
        path_in_repo="eval_results.json",
        repo_id=HF_REPO,
    )

    hf_url = f"https://huggingface.co/{HF_REPO}"
    wandb.run.summary["huggingface_model"] = hf_url
    print(f"Better model pushed to: {hf_url}")
else:
    print(f"Skipping push — existing model ({existing_accuracy:.4f}) is better or equal")

Current run accuracy: 0.9130
Existing HF model accuracy: 0.0000


README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Better model pushed to: https://huggingface.co/somnathchakraborty/distilbert-imdb-sentiment


In [14]:
wandb.finish()
print("Done!")

wandb: updating run metadata
wandb: uploading summary, console lines 17-18
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁█▆██
wandb:                 eval/f1 ▁█▆██
wandb:               eval/loss ▂▁▅█▁
wandb:            eval/runtime █▁▅▇▃
wandb: eval/samples_per_second ▁█▄▂▆
wandb:   eval/steps_per_second ▁█▄▂▇
wandb:             train/epoch ▁▃▆▆███
wandb:       train/global_step ▁▃▆▆███
wandb:         train/grad_norm ▁
wandb:     train/learning_rate ▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.913
wandb:                 eval/f1 0.91301
wandb:               eval/loss 0.53161
wandb:            eval/runtime 9.5823
wandb: eval/samples_per_second 104.359
wandb:   eval/steps_per_second 6.575
wandb:       huggingface_model https://huggingface....
wandb:              total_flos 2646698625146880.0
wandb:             train/epoch 4
wandb:       train/global_step 628
wandb:                      +7 ...
wandb: 
wandb: 🚀 View run run-v

Done!
